# Cuaderno de Trabajo: Análisis Longitudinal de Deterioro Metabólico y Comorbilidades
## Adaptación de Algoritmos de Visión por Computador (Analogía SIFT: Puntos Clave y Descriptores Locales con Ventana Fija de 30 Años)

**Objetivo General:**
Analizar la tasa de cambio real en el deterioro metabólico a lo largo de la vida, detectando puntos clave de inflexión estables mediante una ventana deslizante fija de $30$ años (sin normalización de escala 0-1) y extrayendo vectores descriptores locales basados en comorbilidades (CIE-10) para realizar un emparejamiento (*feature matching*) inter-género (Hombres vs. Mujeres).

In [ ]:
# ===============================================================================
# 1. IMPORTACIÓN DE LIBRERÍAS Y CONFIGURACIÓN DEL ENTORNO
# ===============================================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import logging
import traceback as trace
from pathlib import Path

# Configuración de estilos y visualización
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
sns.set_palette("deep")

# Configuración de Logging profesional
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

# Directorio de salida para gráficos y reportes
output_dir = Path('./output_sift')
output_dir.mkdir(parents=True, exist_ok=True)
SAVE_CHARTS = True
YEAR = 2026

logger.info("✅ Entorno de trabajo inicializado correctamente.")

In [ ]:
# ===============================================================================
# 2. CARGA Y FILTRADO DE DATOS
# ===============================================================================
try:
    logger.info("-------INICIO: Carga y Filtrado de Datos Longitudinales-------")
    
    # Simulación o carga real del dataset tabular
    # (Ajustar la ruta según el entorno del usuario)
    data_path = Path('./data/dataset_salud_longitudinal.csv')
    
    if data_path.exists():
        df_raw = pd.read_csv(data_path)
        logger.info(f"📂 Dataset cargado exitosamente. Dimensiones: {df_raw.shape}")
    else:
        # Dataset sintético de respaldo para demostración estructural
        logger.warning("⚠️ No se encontró el archivo físico. Generando entorno sintético de prueba...")
        np.random.seed(42)
        n_samples = 5000
        edades = np.random.randint(18, 90, n_samples)
        sexos = np.random.choice(['Hombre', 'Mujer'], n_samples)
        
        # Simulación de puntaje Z con aceleración en edades avanzadas
        score_z = []
        for e, s in zip(edades, sexos):
            base = (e / 50.0) ** 1.8
            if s == 'Mujer':
                base += 0.3 if e > 50 else 0.0 # Impacto menopáusico simulado
            score_z.append(base + np.random.normal(0, 0.4))
            
        df_raw = pd.DataFrame({
            'Edad': edades,
            'Sex_desc': sexos,
            'Score_Deterioro_Metabolico_Z': score_z
        })
        
    # Filtrado y limpieza básica
    df_filtered = df_raw.dropna(subset=['Edad', 'Score_Deterioro_Metabolico_Z']).copy()
    logger.info(f"🧹 Registros limpios para análisis: {len(df_filtered)}")
    logger.info("-------FIN: Carga y Filtrado de Datos-------")

except Exception as e:
    logger.error(f"❌ Error crítico en la carga de datos: {e}")
    logger.error(trace.format_exc())

In [ ]:
# ===============================================================================
# 3. SEGMENTACIÓN DE POBLACIÓN (HOMBRES VS MUJERES)
# ===============================================================================
try:
    logger.info("-------INICIO: Segmentación Poblacional por Género-------")
    
    df_hombres = df_filtered[df_filtered['Sex_desc'].str.lower().isin(['hombre', 'h', 'masculino'])].copy()
    df_mujeres = df_filtered[df_filtered['Sex_desc'].str.lower().isin(['mujer', 'm', 'femenino'])].copy()
    
    logger.info(f"👥 Población Segmentada:")
    logger.info(f"   - Hombres: {len(df_hombres)} registros")
    logger.info(f"   - Mujeres: {len(df_mujeres)} registros")
    logger.info("-------FIN: Segmentación Poblacional-------")

except Exception as e:
    logger.error(f"❌ Error en la segmentación: {e}")

## 4. Metodología SIFT: Tasa de Cambio Real con Ventana Fija de 30 Años
A diferencia de enfoques que normalizan de $0$ a $1$, aquí aplicaremos una **ventana deslizante fija de $30$ años** ($\Delta t = 30$). La pendiente obtenida mediante regresión lineal local representa la **tasa de cambio real** en unidades físicas de desviación estándar por año de vida ($\Delta Z / \Delta \text{año}$).

In [ ]:
# ===============================================================================
# 5. CÁLCULO DE LA TASA DE CAMBIO REAL (VENTANA FIJA DE 30 AÑOS)
# ===============================================================================
try:
    logger.info("-------INICIO: Cálculo de Tasa de Cambio Real (Ventana Fija 30 años)-------")
    
    TAMANO_VENTANA = 30
    
    def calcular_tasa_cambio_real(df_grupo, nombre_grupo):
        curva = df_grupo.groupby('Edad')['Score_Deterioro_Metabolico_Z'].mean().reset_index()
        curva = curva.sort_values('Edad').reset_index(drop=True)
        
        resultados_ventana = []
        edades_min = curva['Edad'].min()
        edades_max = curva['Edad'].max()
        
        for edad_inicio in range(edades_min, edades_max - TAMANO_VENTANA + 2):
            edad_fin = edad_inicio + TAMANO_VENTANA - 1
            sub_ventana = curva[(curva['Edad'] >= edad_inicio) & (curva['Edad'] <= edad_fin)]
            
            if len(sub_ventana) >= 5:
                x = sub_ventana['Edad'].values
                y = sub_ventana['Score_Deterioro_Metabolico_Z'].values
                
                # Pendiente real en unidades Z/año
                pendiente, intercepto = np.polyfit(x, y, 1)
                
                resultados_ventana.append({
                    'Edad_Inicio': edad_inicio,
                    'Edad_Fin': edad_fin,
                    'Edad_Centro': edad_inicio + (TAMANO_VENTANA // 2),
                    'Tasa_Cambio_Real': pendiente,
                    'Intercepto': intercepto
                })
                
        df_tasa = pd.DataFrame(resultados_ventana)
        logger.info(f"✅ Ventanas de 30 años procesadas para {nombre_grupo}: {len(df_tasa)} ventanas analizadas.")
        return curva, df_tasa

    # Aplicar a ambos grupos
    curva_h, tasas_h = calcular_tasa_cambio_real(df_hombres, "Hombres")
    curva_m, tasas_m = calcular_tasa_cambio_real(df_mujeres, "Mujeres")
    
    logger.info("-------FIN: Cálculo de Tasa de Cambio Real-------")

except Exception as e:
    logger.error(f"❌ Error al calcular la tasa de cambio real: {e}")
    logger.error(trace.format_exc())

In [ ]:
# ===============================================================================
# 6. VISUALIZACIÓN COMPARATIVA DE TASA DE CAMBIO REAL
# ===============================================================================
try:
    logger.info("-------INICIO: Generación de Gráfica de Tasas Reales-------")
    
    plt.figure(figsize=(11, 5))
    plt.plot(tasas_h['Edad_Centro'], tasas_h['Tasa_Cambio_Real'], label='Tasa Real (Hombres - Ventana 30a)', color='#0275d8', linewidth=2)
    plt.plot(tasas_m['Edad_Centro'], tasas_m['Tasa_Cambio_Real'], label='Tasa Real (Mujeres - Ventana 30a)', color='#d9534f', linewidth=2)
    
    plt.axhline(0, color='gray', linestyle=':', alpha=0.7)
    plt.title(f'Tasa de Cambio Real del Deterioro Metabólico (Ventana Fija = {TAMANO_VENTANA} años)', fontsize=13, fontweight='bold')
    plt.xlabel('Edad Central de la Ventana (Años)', fontsize=11)
    plt.ylabel('Tasa de Cambio Real ($\Delta$ Score Z / Año)', fontsize=11)
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.tight_layout()
    
    if SAVE_CHARTS:
        fig_tasa_path = output_dir / f"4_Tasa_Cambio_Real_Ventana30_{YEAR}.png"
        plt.savefig(fig_tasa_path, dpi=300, bbox_inches='tight')
        logger.info(f"✅ Gráfica guardada en: {fig_tasa_path}")
        
    plt.show()
    logger.info("-------FIN: Generación de Gráfica de Tasas Reales-------")

except Exception as e:
    logger.error(f"❌ Error al graficar tasas de cambio: {e}")

In [ ]:
# ===============================================================================
# 7. EXTRACCIÓN DE DESCRIPTORES LOCALES Y EMPAREJAMIENTO INTER-GÉNERO (SIFT MATCHING)
# ===============================================================================
try:
    logger.info("-------INICIO: SIFT Matching Inter-género-------")
    
    # Calcular derivadas sobre la curva promedio para localizar puntos clave (DoG / Inflexión)
    curva_h['Gradiente_Z'] = np.gradient(curva_h['mean'])
    curva_h['Segunda_Derivada'] = np.gradient(curva_h['Gradiente_Z'])
    
    curva_m['Gradiente_Z'] = np.gradient(curva_m['mean'])
    curva_m['Segunda_Derivada'] = np.gradient(curva_m['Gradiente_Z'])
    
    # Punto clave crítico en hombres
    idx_inflexion_h = curva_h['Segunda_Derivada'].abs().idxmax()
    edad_inflexion_h = curva_h.loc[idx_inflexion_h, 'Edad']
    
    # Construcción de vector descriptor local en vecindad de 3 años alrededor del punto clave
    vecindad_desc = 3
    descriptor_hombre = curva_h.loc(
    )[
        (curva_h['Edad'] >= edad_inflexion_h - vecindad_desc) &
        (curva_h['Edad'] <= edad_inflexion_h + vecindad_desc),
        ['mean', 'Gradiente_Z', 'Segunda_Derivada']
    ].values.flatten()
    
    # Buscar el match más cercano en mujeres
    mejor_match_edad = None
    min_distancia = float('inf')
    
    for edad_m in curva_m['Edad']:
        if edad_m - vecindad_desc < curva_m['Edad'].min() or edad_m + vecindad_desc > curva_m['Edad'].max():
            continue
        descriptor_mujer = curva_m.loc[
            (curva_m['Edad'] >= edad_m - vecindad_desc) &
            (curva_m['Edad'] <= edad_m + vecindad_desc),
            ['mean', 'Gradiente_Z', 'Segunda_Derivada']
        ].values.flatten()
        
        if len(descriptor_mujer) == len(descriptor_hombre):
            dist = np.linalg.norm(descriptor_hombre - descriptor_mujer)
            if dist < min_distancia:
                min_distancia = dist
                mejor_match_edad = edad_m
                
    logger.info(f"🎯 Resultado de Emparejamiento SIFT Inter-género:")
    logger.info(f"   - Inflexión crítica en Hombres: {edad_inflexion_h} años")
    logger.info(f"   - Edad equivalente en Mujeres por SIFT Matching: {mejor_match_edad} años (Distancia: {min_distancia:.4f})")
    logger.info("-------FIN: SIFT Matching Inter-género-------")

except Exception as e:
    logger.error(f"❌ Error en SIFT Matching: {e}")
    logger.error(trace.format_exc())